# Stratified NLP Analysis - Topics by Category

This notebook analyzes topic models trained separately for each group (Issue, Company, Sub-issue, etc.).

Key Benefits:
- **Targeted insights**: Discover patterns specific to each category
- **Interpretability**: Understand topic nuances within context
- **Comparability**: Compare how different categories discuss similar issues
- **Actionability**: Find category-specific improvement areas

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path
from tqdm import tqdm
import sys

sys.path.insert(0, '/Users/zhusizhen/Documents/project')

# Set up plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("Libraries loaded successfully")

ModuleNotFoundError: No module named 'pandas'

## 1. Load Stratified Models Metadata

In [ ]:
# Configure the groupby column (change this to analyze different stratifications)
GROUPBY_COLUMN = 'Issue'  # Can also be 'Company', 'Sub-issue', 'State', etc.

stratified_models_dir = Path('/Users/zhusizhen/Documents/project/data/stratified_models')
groupby_dir = stratified_models_dir / GROUPBY_COLUMN.replace(' ', '_')

# Load metadata
metadata_path = groupby_dir / 'models_index.json'
with open(metadata_path, 'r') as f:
    models_metadata = json.load(f)

print(f"Loaded {len(models_metadata)} trained models for groupby column: {GROUPBY_COLUMN}\n")

# Display metadata
for i, meta in enumerate(models_metadata[:10], 1):
    print(f"{i}. {meta['group_name']:<50} | Topics: {meta['n_topics']:<2} | Type: {meta['model_type']:<4} | Samples: {meta['n_samples']:>6,}")

## 2. Load Original Dataset & Model Results

In [ ]:
# Load original data for reference
csv_path = '/Users/zhusizhen/Documents/project/data/complaints-2026-05-17_11_33.csv'
df = pd.read_csv(csv_path)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Show summary of groups
group_counts = df[GROUPBY_COLUMN].value_counts()
print(f"\nGroup distribution ({GROUPBY_COLUMN}):")
print(group_counts.head(10))

## 3. Extract Top Terms for Each Group

In [ ]:
def get_top_terms_for_model(model, vectorizer, n_terms=10):
    """
    Extract top terms for each topic in a model.
    
    Returns: dict mapping topic_id -> list of (word, weight) tuples
    """
    feature_names = vectorizer.get_feature_names_out()
    topics = {}
    
    for topic_id, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_terms:][::-1]
        top_terms = [(feature_names[idx], topic[idx]) for idx in top_indices]
        topics[topic_id] = top_terms
    
    return topics

# Load models and extract top terms
group_topics = {}  # Maps group_name -> {topic_id -> [(word, weight), ...]}

for meta in tqdm(models_metadata, desc='Loading models'):
    group_name = meta['group_name']
    model_path = meta['model_path']
    vec_path = meta['vectorizer_path']
    
    try:
        model = joblib.load(model_path)
        vectorizer = joblib.load(vec_path)
        
        top_terms = get_top_terms_for_model(model, vectorizer, n_terms=10)
        group_topics[group_name] = {
            'metadata': meta,
            'topics': top_terms
        }
    except Exception as e:
        print(f"Error loading {group_name}: {e}")

print(f"\nSuccessfully loaded {len(group_topics)} model groups")

## 4. Display Topics by Group

In [ ]:
# Show topics for first few groups
for i, (group_name, group_data) in enumerate(list(group_topics.items())[:3]):
    meta = group_data['metadata']
    topics = group_data['topics']
    
    print(f"\n{'='*80}")
    print(f"GROUP: {group_name}")
    print(f"Model: {meta['model_type']} with {meta['n_topics']} topics")
    print(f"Samples: {meta['n_samples']:,} | Score: {meta['score']:.4f}")
    print(f"{'='*80}")
    
    for topic_id, terms in topics.items():
        print(f"\nTopic {topic_id}:")
        terms_str = ' | '.join([f"{word}({weight:.2f})" for word, weight in terms[:7]])
        print(f"  {terms_str}")

## 5. Visualize Topics for Each Group

In [ ]:
# Create visualization for each group (first 5)
for group_name, group_data in list(group_topics.items())[:5]:
    meta = group_data['metadata']
    topics = group_data['topics']
    
    n_topics = len(topics)
    cols = 2
    rows = (n_topics + 1) // 2
    
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3*rows))
    axes = axes.flatten()
    
    for topic_id, terms in topics.items():
        ax = axes[topic_id]
        
        words = [w for w, _ in terms[:10]]
        weights = [wt for _, wt in terms[:10]]
        
        ax.barh(words, weights, color='steelblue')
        ax.set_xlabel('Weight')
        ax.set_title(f'Topic {topic_id}')
        ax.invert_yaxis()
    
    # Hide extra subplots
    for idx in range(n_topics, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f"{GROUPBY_COLUMN}: {group_name} ({meta['n_samples']:,} complaints, {meta['n_topics']} topics)",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Compare Topic Vocabularies Across Groups

In [ ]:
# Extract all unique words across all groups and their importance
word_importance = {}  # word -> {group_name -> avg_weight}

for group_name, group_data in group_topics.items():
    topics = group_data['topics']
    for topic_id, terms in topics.items():
        for word, weight in terms:
            if word not in word_importance:
                word_importance[word] = {}
            word_importance[word][group_name] = max(
                word_importance[word].get(group_name, 0), weight
            )

# Find words that appear frequently across groups
word_group_count = {w: len(groups) for w, groups in word_importance.items()}
common_words = sorted(word_group_count.items(), key=lambda x: x[1], reverse=True)[:20]

print(f"Top 20 Words Appearing Across Groups:")
print(f"{'Word':<20} {'Groups':<10}")
print("-" * 30)
for word, count in common_words:
    print(f"{word:<20} {count:<10}")

## 7. Sample Complaints from Each Topic in Each Group

In [ ]:
def get_sample_complaints_for_group(group_name, model, vectorizer, df, n_samples=2):
    """
    Get sample complaints for each topic in a group.
    """
    from utils.preprocessing import preprocess_batch
    
    # Filter data for this group
    group_df = df[df[GROUPBY_COLUMN] == group_name].copy()
    
    # Preprocess texts
    processed = group_df['Consumer complaint narrative'].apply(
        lambda x: ' '.join(preprocess_batch([str(x)])[0]) if pd.notna(x) else ''
    )
    
    # Vectorize
    X = vectorizer.transform(processed)
    
    # Get topic predictions
    topic_dist = model.transform(X)
    primary_topics = topic_dist.argmax(axis=1)
    
    samples = {}  # topic_id -> list of sample texts
    
    for topic_id in range(model.n_components_):
        topic_indices = np.where(primary_topics == topic_id)[0]
        if len(topic_indices) == 0:
            samples[topic_id] = []
        else:
            # Get highest confidence samples
            confidences = topic_dist[topic_indices, topic_id]
            top_indices = topic_indices[confidences.argsort()[-n_samples:][::-1]]
            samples[topic_id] = group_df.iloc[top_indices]['Consumer complaint narrative'].tolist()
    
    return samples

# Get samples for first group
first_group = list(group_topics.keys())[0]
first_meta = group_topics[first_group]['metadata']

print(f"Fetching sample complaints for: {first_group}")

model = joblib.load(first_meta['model_path'])
vectorizer = joblib.load(first_meta['vectorizer_path'])

samples = get_sample_complaints_for_group(first_group, model, vectorizer, df, n_samples=1)

print(f"\n{'='*80}")
print(f"SAMPLE COMPLAINTS - {first_group}")
print(f"{'='*80}")

for topic_id, complaints in samples.items():
    if complaints:
        print(f"\nTopic {topic_id}:")
        complaint = complaints[0][:300] + "..." if len(complaints[0]) > 300 else complaints[0]
        print(f"  {complaint}")

## 8. Summary Statistics

In [ ]:
# Create summary statistics
summary = []

for group_name, group_data in group_topics.items():
    meta = group_data['metadata']
    summary.append({
        GROUPBY_COLUMN: group_name,
        'Model Type': meta['model_type'],
        'Topics': meta['n_topics'],
        'Complaints': meta['n_samples'],
        'Score': meta['score']
    })

summary_df = pd.DataFrame(summary)
summary_df = summary_df.sort_values('Complaints', ascending=False)

print(f"\nSTRATIFIED MODELS SUMMARY (Grouped by {GROUPBY_COLUMN})\n")
print(summary_df.to_string(index=False))
print(f"\nTotal Groups: {len(summary_df)}")
print(f"Total Complaints Covered: {summary_df['Complaints'].sum():,}")